Calculate $P(M_{\rm{UV}}|M_h)$, make sure it integrates to 1

In [200]:
from zeus21 import constants
from zeus21 import sfrd
import zeus21
from scipy.special import erf
from escripts import eMCMC

In [242]:
# Purpose: interface with Zeus to run MCMC on models of the UVLF with time-evolving sigma_UV
# Author: Emily Bregou, Julian Muñoz

# Standard packages
import numpy as np
import emcee
import pandas as pd

# Local packages
import zeus21

# MCMC class
class UVLF():
    def __init__(self, data, param_data = None, MINRELERROR = 0.2):
        self.data = data
        if type(self.data[0]) is not list: # Make sure the format is correct for the rest of the script if only one redshift is input
            self.data = [self.data] 
        self.zs = [dat[0] for dat in self.data]
        if param_data is None:
            param_data = get_default_dict(table=False)
        self.UserParams = zeus21.User_Parameters()
        self.param_data = param_data
        self.lowers = [p['lower'] for p in list(self.param_data.values()) if p.get('fit', True)]
        self.uppers = [p['upper'] for p in list(self.param_data.values()) if p.get('fit', True)]
        self.ndim = len(self.lowers)
        self.nwalkers = 2*self.ndim # Walkers = twice the number of parameters
        self.MINRELERROR = MINRELERROR

        # Get cosmological parameters, construct HMF from Zeus
        CosmoParams_input = zeus21.Cosmo_Parameters_Input(zmin_CLASS=0.0)
        self.CosmoParams,ClassyCosmo, CorrFclass, self.HMFintclass =  zeus21.cosmo_wrapper(self.UserParams, CosmoParams_input)

    def UVLF_wrapper(self, zcenter, zwidth, MUVcenters, MUVwidths, paramvector):
        """
        Computes and returns the UVLF at z=zcenters, with width zwidths, in bins centered at MUVcenters with width MUVwidths
        Inputs:
            zcenters [float]: center redshift value for binned UVLF data
            zwdith [float]: width of the redshift bin
            MUVcenters [1darray]: the central UV magnitude in each bin
            MUVwidths [1darray]: the width of the UV magnitude bins
            paramvector [1darray]: parameters
        Outputs:
            PhiUV [1darray]: In units of mag^-1 Mpc^-3
        """

        params = self.time_evolution(paramvector, zcenter)
        astroparams = self.param_wrapper(params)
        UVLFs_std = UVLF_binned(astroparams,self.CosmoParams,self.HMFintclass,zcenter,zwidth,MUVcenters,MUVwidths)
        
        return UVLFs_std

    def time_evolution(self, paramvector, zcenter):
        """
        Applies the time evolution of each parameter so that we feed the evolved value, matching the given redshift, to the UVLF wrapper
        Inputs:
            paramvector [1darray]: parameters
            zcenter [float]: center redshift value for binned UVLF data
        Returns:
            [log10epsstar, log10Mcstar, alphastar, betastar, sigmaUV]: values of these parameters that match the given redshift
        """
        param_data = list(self.param_data.values())
        assert len(paramvector) == len([p for p in param_data if p['fit']]), 'The length of paramvector does not match the number of parameters you want to fit'
        
        # Deal with constant parameters, deal with piecewise
        params = np.zeros(len(param_data))
        j = 0 # This keeps track of how far we are into paramvector
        for i, param in enumerate(param_data): 
            if param['fit']: # If this is a fit parameter
                value = paramvector[j] 
                j+=1
            else: # If this parameter is held constant
                value = param['value']

            if not np.isscalar(value): # Piecewise, get the value that corresponds to that redshift
                index = np.where(self.zs == zcenter)[0][0]
                value = value[index]
                
            params[i] = value

        # Apply time evolution
        final_values = []
        for i in range(5): # This is alpha*, beta*, M_h, eps*, sigmaUV, the 5 base parameters
            base_idx = 2 * i
            value = params[base_idx]
            time_deriv = params[base_idx+1] # The order of the parameters are alternating base & time derivative, so this works
            final_values.append(value+(time_deriv*(zcenter-8)))

        # Apply mass dependence of sigma
        sig = final_values[-1]
        dsigdM = params[-1]
        final_values[-1] = sig + (dsigdM*(np.log10(self.HMFintclass.Mhtab)-10))

        return final_values

    def param_wrapper(self, params):
        """
        Puts paramvector into a format that Zeus can read
        Inputs:
            params [1darray]: parameters
        Outputs:
            astroparams [zeus Astro_Parameters object]: parameters for the UVLF, wrapped so that Zeus can read them
        """
        alphastar, betastar, log10Mcstar, log10epsstar, sigmaUV = params
        astroparams = zeus21.Astro_Parameters(self.UserParams, self.CosmoParams, epsstar=10**log10epsstar, Mc=10**log10Mcstar,
                                              alphastar=alphastar, 
                                              betastar=betastar, sigmaUV = sigmaUV) 
        
        return astroparams


def build_param_data(custom_params):
    """
    Builds the metadata around each parameter. You need only specify the parameters & the keys you want to change; anything unspecified will assume
    its default value. The order in which you pass the modifcations is not important, as long as you match the keys.
    Inputs:
        custom_params [dict of dicts]: nested dictionary that describes how you want to modify parameters from their default values
    Returns:
        default_values [dict of dicts]: dictionary with updated data set by custom_params and default data otherwise                        
    """
    
    # Master dictionary with defaults for each parameter label
    default_values = get_default_dict(table = False)

    if custom_params is None:
        return default_values
        
    for label in custom_params:
        if label not in default_values:
            raise ValueError(f"No default values found for label: {label}")

        default_values[label].update(custom_params[label])

    return default_values

def get_default_dict(table = True):
    """
    All supported parameters are included in this nested dictionary, including the following keys:
    'fit': when True, this will be fit with MCMC. When False, this value will be held fixed at the value provided under the 'value' key.
    'value': value to assign this parameter when 'fit' is False
    'start': starting value for this parameter in the MCMC
    'lower': lower bound for the flat prior in MCMC
    'upper': upper bound for the flat prior in MCMC
    'label': label, in mathematical format, for plotting
    Note that you may assign the 'value' of base parameters (alpha, beta, logMc, loge, sig) to be arrays. The code will interpret these as
    piecewise values across redshift (so the length of the array must match the number of redshifts you have data to fit for). This will only work
    if 'fit' is labeled False; there is currently not support for using MCMC to fit parameters in a piecewise fashion. 

    If table is true, it will return a readable pandas dataframe. If not, it will return as a dictionary.
    """
    default_values = {
        'alpha':   {'fit': True, 'value': 0.6, 'start': 0.6, 'lower': 0, 'upper': 4, 'label': r"$\alpha$"},
        'dalphadz':{'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 'label': r"$d\alpha/dz$"},
        'beta':    {'fit': True, 'value': -0.5, 'start': -0.5, 'lower': -1, 'upper': 0, 'label': r"$\beta$"},
        'dbetadz': {'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 'label': r"$d\beta/dz$"},
        'logMc':   {'fit': True, 'value': 12, 'start': 12, 'lower': 9, 'upper': 16, 'label': r'$\log(M_c)$'},
        'dlogMcdz':{'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 'label': r'$d\log(M_c)/dz$'},
        'loge':    {'fit': True, 'value': -0.5, 'start': -1, 'lower': -1, 'upper': 1, 'label': r"$\log(\epsilon_0)$"},
        'dlogedz': {'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 'label': r'$d\log(\epsilon_0)/dz$'},
        'sig':     {'fit': True, 'value': 0, 'start': 0.5, 'lower': 0, 'upper': 6, 'label': r'$\sigma_{\rm{UV}, 10}$'},
        'dsigdz':  {'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 'label': r'$d\sigma_{\rm{UV}}/dz$'},
        'dsigdM':  {'fit': True, 'value': 0, 'start': 0.01, 'lower': -0.5, 'upper': 0.5, 
                    'label': r'$d\sigma_{\rm{UV}}/dM_{h}$'}
    }

    if table:
        return pd.DataFrame(default_values).T
    else:
        return default_values

def make_table(best_fits, param_labels, fit_labels):
    """
    Make a table to compare best fit values of parameters
    Inputs:
        best_fits [list of lists]: list of best fit parameters
        param_labels [list of strs]: names of parameters (rows of the table)
        fit_labels [list]: names of each type of fit (columns of the table)
    Outputs:
        dataframe table with labeled parameters for comparison
    """
    df = pd.DataFrame(best_fits, columns = param_labels)
    df.index = fit_labels
    
    return df.T

In [412]:
# Get data
datBouw21 = np.loadtxt('/Users/eb35267/Desktop/code/home/data/Bouwens21_fixed.txt',  skiprows=2, unpack=True)
redshiftsBouw21 = np.unique(datBouw21[0])
dredshiftsBouw21 = np.ones_like(redshiftsBouw21)/2. #approximate, there are true window functions to use

datHST = [] 
#format is     zdat = data[0] zerr = data[1] xdat = data[2],  ydat = data[3]  yerr = data[4]  xerr = data[5] 

for iz,z in enumerate(redshiftsBouw21): # Create a bunch of arrays that contain all of the data listed above. One for each redshift
    zlistindex = datBouw21[0] == 1.0*z
    datarr = [z,dredshiftsBouw21[iz], datBouw21[1][zlistindex], datBouw21[3][zlistindex], datBouw21[4:6][:,zlistindex], datBouw21[2][zlistindex]]
    datHST= datHST + [datarr]

In [413]:
datHST

[[4.0,
  0.5,
  array([-22.69, -22.19, -21.69, -21.19, -20.69, -20.19, -19.69, -19.19,
         -18.69, -17.94, -16.94, -15.94]),
  array([5.000e-06, 1.500e-05, 1.440e-04, 3.440e-04, 6.980e-04, 1.624e-03,
         2.276e-03, 3.056e-03, 4.371e-03, 1.016e-02, 2.742e-02, 2.882e-02]),
  array([[ 2.000e-06,  4.500e-06,  1.100e-05,  1.900e-05,  3.400e-05,
           6.550e-05,  9.950e-05,  1.940e-04,  3.445e-04,  4.600e-04,
           1.720e-03,  4.370e-03],
         [-2.000e-06, -4.500e-06, -1.100e-05, -1.900e-05, -3.400e-05,
          -6.550e-05, -9.950e-05, -1.940e-04, -3.445e-04, -4.600e-04,
          -1.720e-03, -4.370e-03]]),
  array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])],
 [5.0,
  0.5,
  array([-23.11, -22.61, -22.11, -21.61, -21.11, -20.61, -20.11, -19.61,
         -19.11, -18.36, -17.36, -16.36]),
  array([1.000e-06, 4.000e-06, 2.800e-05, 9.200e-05, 2.620e-04, 5.840e-04,
         8.790e-04, 1.594e-03, 2.159e-03, 4.620e-03, 8.780e-03, 2.512e-02]),
  array([[ 5

In [392]:
def UVLF_binned(Astro_Parameters,Cosmo_Parameters,HMF_interpolator, zcenter, zwidth, MUVcenters, MUVwidths, DUST_FLAG=True, RETURNBIAS = False):
    'Binned UVLF in units of 1/Mpc^3/mag, for bins at <zcenter> with a Gaussian width zwidth, centered at MUV centers with tophat width MUVwidths. z width only in HMF since that varies the most rapidly. If flag RETURNBIAS set to true it returns number-avgd bias instead of UVLF, still have to divide by UVLF'
    
    if(constants.NZ_TOINT>1):
        DZ_TOINT = np.linspace(-np.sqrt(constants.NZ_TOINT/3.),np.sqrt(constants.NZ_TOINT/3.),constants.NZ_TOINT) #in sigmas around zcenter
        # Dimensionless
        # The number of steps around the center redshift to take is constants.NZ_TOINT, specified in constants.py 
    else:
        DZ_TOINT = np.array([0.0])
    WEIGHTS_TOINT = np.exp(-DZ_TOINT**2/2.)/np.sum(np.exp(-DZ_TOINT**2/2.)) # Get the weight that would correspond to each dz, assuming Gaussian
    # distribution around central value


    # Because the range of redshifts is only applied to the HMF, we get one value of SFR
    SFRlist = sfrd.SFR_II(Astro_Parameters,Cosmo_Parameters,HMF_interpolator, HMF_interpolator.Mhtab, zcenter, zcenter) # SFR based off of halo mass
    sigmaUV = Astro_Parameters.sigmaUV # Set sigma at initialization (fixed value, redshift dependence is handled by a wrapper function)
  
    
    if (constants.FLAG_RENORMALIZE_LUV == True): #lower the LUV (or SFR) to recover the true avg, not log-avg
        SFRlist/= np.exp((np.log(10)/2.5*sigmaUV)**2/2.0) # almost always want to ignore this
        
    
    MUVbarlist = zeus21.UVLFs.MUV_of_SFR(SFRlist, Astro_Parameters._kappaUV) #avg for each Mh
    # kappaUV is set to a constant in input.py & comes from Madau+Dickinson14
    MUVbarlist = np.fmin(MUVbarlist,constants._MAGMAX) # fmin -> piecewise minimum
    # Make sure none of the MUVs exceed the maximum value (set to make sure there aren't errors or NaN)
    
    
    if(RETURNBIAS==True): # weight by bias. As it says in the docstring, this is the bias times the UVLF, so you still need to divide by the UVLF
        # (see eq. 6 in Muñoz+23)
        biasM = np.array([bias_Tinker(Cosmo_Parameters, HMF_interpolator.sigma_int(HMF_interpolator.Mhtab,zcenter+dz*zwidth)) for dz in DZ_TOINT])
    else: # do not weight by bias (if you just want the UVLF)
        biasM = np.ones_like(WEIGHTS_TOINT)
 
        
    HMFtab = np.array([HMF_interpolator.HMF_int(HMF_interpolator.Mhtab,zcenter+dz*zwidth) for dz in DZ_TOINT]) # Get HMF for each given z
    HMFcurr = np.sum(WEIGHTS_TOINT * HMFtab.T * biasM.T,axis=1) #biasM is 1 unless RETURNBIAS is True

    #cannot directly 'dust' the theory since the properties of the IRX-beta relation are calibrated on observed MUV. Recursion instead:
    currMUV = MUVbarlist
    if(DUST_FLAG==True):
        currMUV2 = np.ones_like(currMUV)
        while(np.sum(np.abs((currMUV2-currMUV)/currMUV)) > 0.02):
            currMUV2 = currMUV
            currMUV = MUVbarlist + zeus21.UVLFs.AUV(Astro_Parameters,zcenter,currMUV)
           
    # Apply sigmaUV & bin the UVLF to match observational data (both of these things are happening at once, but they are differnet things!):
    
    # Define each observational MUV bin
    MUVcuthi = MUVcenters +  MUVwidths/2.
    MUVcutlo = MUVcenters -  MUVwidths/2.

    xhi = np.subtract.outer(MUVcuthi , currMUV)/(np.sqrt(2) * sigmaUV) # Create Gaussian around each element of currMUV & determine what part
                                                                        # of those Gaussians falls in each bin (range of xhi to xlo)
    xlo = np.subtract.outer(MUVcutlo, currMUV )/(np.sqrt(2) * sigmaUV)
    weights = (erf(xhi) - erf(xlo)).T/(2.0 * MUVwidths) # Integrate the contribution of each P(MUV|Mh) to each bin in MUV; weights is 
                                                        # the integral of binned P(MUV|Mh) dMUV (tophat binning), see notes from 6/10/25
    
    return [erf(xhi)[i][-1] for i in range(len(xhi))], [erf(xlo)[i][-1] for i in range(len(xlo))]

In [393]:
eMCMC.get_default_dict()

,fit,value,start,lower,upper,label
alpha,True,0.6,0.6,0,4,$\alpha$
dalphadz,True,0,0.01,-0.5,0.5,$d\alpha/dz$
beta,True,-0.5,-0.5,-1,0,$\beta$
dbetadz,True,0,0.01,-0.5,0.5,$d\beta/dz$
logMc,True,12,12,9,16,$\log(M_c)$
dlogMcdz,True,0,0.01,-0.5,0.5,$d\log(M_c)/dz$
loge,True,-0.5,-1,-1,1,$\log(\epsilon_0)$
dlogedz,True,0,0.01,-0.5,0.5,$d\log(\epsilon_0)/dz$
sig,True,0,0.5,0,6,"$\sigma_{\rm{UV}, 10}$"
dsigdz,True,0,0.01,-0.5,0.5,$d\sigma_{\rm{UV}}/dz$


In [394]:
param_dict = eMCMC.build_param_data({})
my_UVLF = UVLF(datHST, param_dict)

In [432]:
paramvector = [0.9, 0, -0.1, 0, 12, 0, -0.5, 0, 0.5, 0, 0.2]

In [433]:
my_UVLF.UVLF_wrapper(4, 0.5, datHST[0][2], datHST[0][5], paramvector)

12
35


([0.992123747001919,
  0.9976494470228836,
  0.9993887208469628,
  0.9998616922943119,
  0.9999728078570425,
  0.9999953592573017,
  0.9999993130648238,
  0.9999999118690398,
  0.9999999902056442,
  0.9999999997230755,
  0.9999999999985627,
  0.9999999999999958],
 [0.9769565580711685,
  0.992123747001919,
  0.9976494470228836,
  0.9993887208469628,
  0.9998616922943119,
  0.9999728078570425,
  0.9999953592573017,
  0.9999993130648238,
  0.9999999118690398,
  0.9999999969065323,
  0.9999999999785492,
  0.9999999999999167])

In [385]:
len([0.5356371866380181,
  0.6484990001410593,
  0.7422317601285281,
  0.8170352245376713,
  0.8743990219049799,
  0.9166697654389777,
  0.9466012063147179,
  0.9669669852130565,
  0.9802825625318883,
  0.9915036601046399,
  0.9975680457021139,
  0.9994001490540364])

12

In [401]:
test = np.array([[5,6], [7,8]])

In [402]:
test

array([[5, 6],
       [7, 8]])

In [403]:
test/5

array([[1. , 1.2],
       [1.4, 1.6]])